In [ ]:
#| hide
from compact.core import *
from compact import lisp
from fasthtml.common import *

# Example blog

> Implement a blog using embedded lisp in python with fasthtml

take a look at llms.txt at /app/data/compact/llms.txt

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

- ⏳ `py(code="view_file('/app/data/compact/llms.txt')")` ⏳


````json {.tool}
{
  "id": "chatcmpl-tool-94bdde388cccd2fa",
  "name": "py",
  "args": {
    "code": "view_file('/app/data/compact/llms.txt')"
  },
  "result": "1: # compact\n2: \n3: > A tiny Scheme interpreter embedded in Python with a shared namespace. Evaluate Scheme from Python via `\"(expr)\" @ lisp`; Python variables are visible in Lisp, and anything `define`d in Lisp lands back in Python.\n4: \n5: `pip install compact-scheme`. Supports: define, set!, lambda, if, cond, case, and, or, begin, quote, let, let*, letrec (named let), quasiquote (` , ,@), macro, when, unless, fold-left, fold-right, map, filter, for-each, apply.\n6: \n7: Primitives: `+ - * / = < > <= >= abs min max sqrt expt modulo remainder floor ceiling round truncate cons car cdr list null? pair? list? equal? string-append string-length substring number->string string->number symbol->string string->symbol number? string? symbol? boolean? not error`\n8: \n9: ```python\n10: from compact import lisp\n11: a, b = 3, 4\n12: \"(define c (sqrt (+ (* a a) (* b b))))\" @ lisp  # c is now 5.0 in Python\n13: ```\n14: \n15: ## Docs\n16: \n17: - [compact documentation](https://thechandru.github.io/compact/): full docs with examples\n18: - [Source on GitHub](https://github.com/thechandru/compact): notebooks and implementation\n19: \n20: ## Examples\n21: \n22: - [Quick start and interop](https://thechandru.github.io/compact/): @ lisp operator, define round-trip, TCO, and macros\n23: \n24: ## Optional\n25: \n26: - Tail calls are optimized (TCO) — deep recursion is safe\n27: - Python callables are directly callable from Lisp and vice versa\n28: - `lisp['hyphen-name']` looks up symbols with hyphens (invalid as Python identifiers)\n29: - Primitives cannot be redefined"
}
````





This is the `llms.txt` for the `compact` library — a tiny Scheme interpreter embedded in Python with bidirectional variable sharing. Key points:

- **Interop via `@ lisp`**: evaluate Scheme from Python (`"(expr)" @ lisp`), with Python variables visible in Lisp and Lisp `define`s landing back in Python
- **Supported forms**: `define`, `set!`, `lambda`, `if`, `cond`, `case`, `let`/`let*`/`letrec`, quasiquote, macros, `map`, `filter`, `fold`, etc.
- **TCO**: tail calls are optimized, so deep recursion is safe
- **Python callables** work directly from Lisp and vice versa

This is the library being used in the blog example above, where Scheme code blocks are embedded in Python via `"""...""" @ lisp` to define posts, routes, and rendering functions for a FastHTML app.

```json {.usage}
{"model": "accounts/fireworks/models/glm-5p2", "prompt_tokens": 71369, "completion_tokens": 297, "total_tokens": 71666, "cached_tokens": 35430, "cost": 0.0848724}
```


we want to build the blog that is described in the note above using compact

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The note already contains a complete Compact-based blog implementation. It uses:

- **Compact/Scheme** for data structures, macros, posts, and routes
- **FastHTML** for rendering and serving
- A shared Scheme environment so definitions persist across blocks

To run it:

1. Install the dependencies:

```bash
pip install compact-scheme python-fasthtml
```

2. Run the code in the note from top to bottom.

3. Open the local URL printed by `serve()`.

The blog defines:

- `/` — an index listing all posts
- `/post/recursion` — the recursion post
- `/post/lists` — the lists post

The main design is the `post` macro:

```scheme
(post "recursion" "On Recursion" "2024-01-10"
  (prose "Recursion is defining something in terms of itself.")
  (example (fact 10)))
```

That declaration constructs a post, registers it, and renders its body through the shared FastHTML environment. The `get` macro similarly turns concise Scheme route declarations into FastHTML routes.

One consistency fix is needed before running: use the same Compact import throughout. The `llms.txt` documents:

```python
from compact import lisp
```

So the Python setup should use that import rather than mixing `compact.core`, `compact`, and `compact.scheme` unless those modules are specifically required by the installed version.

```json {.usage}
{"model": "gpt-5.6-luna", "prompt_tokens": 40759, "completion_tokens": 640, "total_tokens": 41399, "reasoning_tokens": 334, "cost": 0.013379699999999998}
```


In [ ]:
from fasthtml.common import *
from fasthtml.jupyter import *

from compact import lisp

In [ ]:
app,rt = fast_app(title="blog.scm")

shall we build and display a single tag using lisp?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes. Start with a single paragraph tag created by Lisp and rendered by FastHTML:

```python
tag = '(P "Hello from Lisp!")' @ lisp
show(tag)
```

`P` is the FastHTML paragraph constructor, and `@ lisp` evaluates the Scheme expression. `show(tag)` displays the resulting HTML component in the notebook.

To inspect the generated component instead:

```python
tag
```

```json {.usage}
{"model": "gpt-5.6-luna", "prompt_tokens": 41230, "completion_tokens": 195, "total_tokens": 41425, "cached_tokens": 40716, "reasoning_tokens": 103, "cost": 0.00172668}
```


In [ ]:
"""
(P "hello from scheme")
""" @ lisp

<p>hello from scheme</p>

In [ ]:
str(
    """
    (A "a sample post" (href "/posts/a-sample-post"))
    """ @lisp
)

KeyError: 'href'

so how do we start here? should we start with a data structure?


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes. Start with a small data structure for one post. A tagged list is enough:

```python
"""
(define (make-post slug title date body)
  (list 'post slug title date body))

(define post-slug  (lambda (p) (cadr p)))
(define post-title (lambda (p) (caddr p)))
(define post-date  (lambda (p) (cadddr p)))
(define post-body  (lambda (p) (car (cddddr p))))

(define my-post
  (make-post "hello" "Hello, Lisp!" "2026-08-03"
             (P "This is my first post.")))
""" @ lisp
```

The post stores four fields:

```text
slug, title, date, body
```

The `'post` tag lets us recognize the kind of value later. The selectors hide the internal list layout, so the rest of the blog does not need to know how posts are represented.

Run that first and inspect `my_post` in Python.

```json {.usage}
{"model": "gpt-5.6-luna", "prompt_tokens": 41456, "completion_tokens": 415, "total_tokens": 41871, "cached_tokens": 41187, "reasoning_tokens": 191, "cost": 0.00206331}
```


In [ ]:
"""
(define (make-post slug title date body)
  (list 'post slug title date body))

(define my-post
  (make-post "hello"
             "Hello, Lisp!"
             "2026-08-03"
             (P "This is my first post.")))
""" @ lisp

'my-post'

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Now give the post a way to reveal its fields. Define one selector, then use it:

```python
"""
(define (post-title p) (caddr p))
(post-title my-post)
""" @ lisp
```

The result should be:

```text
Hello, Lisp!
```

This confirms that `my-post` is a tagged list and that the title is stored in the expected position.

```json {.usage}
{"model": "gpt-5.6-luna", "prompt_tokens": 42066, "completion_tokens": 152, "total_tokens": 42218, "cached_tokens": 41698, "reasoning_tokens": 63, "cost": 0.00163494}
```


should this be in python ?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The **cell wrapper is Python**, but the **data structure itself is defined in Lisp**:

```python
"""
(define (make-post slug title date body)
  (list 'post slug title date body))

(define my-post
  (make-post "hello"
             "Hello, Lisp!"
             "2026-08-03"
             (P "This is my first post.")))
""" @ lisp
```

Python is handling the triple-quoted string and sending it to Compact. Everything inside the string is Scheme.

Use Python for:

- imports and application setup
- `fast_app(...)`
- serving the app
- inspecting or displaying results

Use Lisp for:

- post data structures
- constructors and selectors
- blog-specific logic and macros

So this should go in a **Python code cell**, just like your earlier `"""...""" @ lisp` example.

```json {.usage}
{"model": "gpt-5.6-luna", "prompt_tokens": 41741, "completion_tokens": 265, "total_tokens": 42006, "cached_tokens": 41413, "reasoning_tokens": 77, "cost": 0.00181779}
```


In [ ]:
"""
(define (caar  p) (car (car p)))
(define (cadr  p) (car (cdr p)))
(define (cdar  p) (cdr (car p)))
(define (cddr  p) (cdr (cdr p)))

(define (caaar p) (car (car (car p))))
(define (caadr p) (car (car (cdr p))))
(define (cadar p) (car (cdr (car p))))
(define (caddr p) (car (cdr (cdr p))))
(define (cdaar p) (cdr (car (car p))))
(define (cdadr p) (cdr (car (cdr p))))
(define (cddar p) (cdr (cdr (car p))))
(define (cdddr p) (cdr (cdr (cdr p))))

(define (cadddr p) (car (cdr (cdr (cdr p)))))
(define (cddddr p) (cdr (cdr (cdr (cdr p)))))
""" @ lisp

'cddddr'

In [ ]:
"""
(define (post-title p) (caddr p))
(post-title my-post)
""" @ lisp

'Hello, Lisp!'

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The selector works, so the post structure is behaving correctly. Add the remaining selectors and a predicate:

```python
"""
(define (post-slug p) (cadr p))
(define (post-date p) (cadddr p))
(define (post-body p) (car (cddddr p)))

(define (post? p)
  (and (pair? p) (eq? (car p) 'post)))
""" @ lisp
```

Then test the abstraction:

```python
"""
(list (post? my-post)
      (post-slug my-post)
      (post-date my-post)
      (post-body my-post))
""" @ lisp
```

This keeps the internal tagged-list representation hidden behind named operations.

```json {.usage}
{"model": "gpt-5.6-luna", "prompt_tokens": 42513, "completion_tokens": 220, "total_tokens": 42733, "cached_tokens": 41413, "reasoning_tokens": 60, "cost": 0.00196839}
```


In [ ]:
"""
(define (post-slug p) (cadr p))
(define (post-date p) (cadddr p))
(define (post-body p) (car (cddddr p)))

(define (post? p)
  (and (pair? p) (equal? (car p) 'post)))
""" @ lisp

'post?'

In [ ]:
"""
(list (post? my-post)
      (post-slug my-post)
      (post-date my-post)
      (post-body my-post))
""" @ lisp

[True, 'hello', '2026-08-03', <p>This is my first post.</p>]

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The abstraction is working: `post?` returns `True`, and the selectors retrieve each field, including the rendered `<p>` body.

Next, create a collection and register the post:

```python
"""
(define *posts* '())

(define (register-post p)
  (set! *posts* (cons p *posts*)))

(register-post my-post)
""" @ lisp
```

`*posts*` stores posts in reverse insertion order because `cons` adds to the front. Later, `reverse` can restore publication order.

```json {.usage}
{"model": "gpt-5.6-luna", "prompt_tokens": 42931, "completion_tokens": 245, "total_tokens": 43176, "cached_tokens": 42470, "reasoning_tokens": 127, "cost": 0.0018533999999999998}
```


In [ ]:
"""
(define *posts* '())

(define (register-post p)
  (set! *posts* (cons p *posts*)))

(register-post my-post)
""" @ lisp

'*posts*'

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The post is now registered in `*posts*`. Add functions to retrieve one post by slug or all posts in publication order:

```python
"""
(define (find-post slug)
  (define (search ps)
    (cond ((null? ps) #f)
          ((equal? (post-slug (car ps)) slug) (car ps))
          (else (search (cdr ps)))))
  (search *posts*))

(define (all-posts)
  (reverse *posts*))
""" @ lisp
```

Test the two operations:

```python
"""
(list (post-title (find-post "hello"))
      (map post-title (all-posts)))
""" @ lisp
```

The expected result is:

```text
["Hello, Lisp!", ["Hello, Lisp!"]]
```

`find-post` searches by the public slug, while `all-posts` hides the internal reverse-storage detail.

```json {.usage}
{"model": "gpt-5.6-luna", "prompt_tokens": 43161, "completion_tokens": 232, "total_tokens": 43393, "cached_tokens": 42888, "reasoning_tokens": 35, "cost": 0.0017861399999999999}
```


In [ ]:
"""
(define (reverse xs)
  (define (loop remaining result)
    (if (null? remaining)
        result
        (loop (cdr remaining)
              (cons (car remaining) result))))
  (loop xs '()))
""" @ lisp

'reverse'

In [ ]:
"""
(define (find-post slug)
  (define (search ps)
    (cond ((null? ps) #f)
          ((equal? (post-slug (car ps)) slug) (car ps))
          (else (search (cdr ps)))))
  (search *posts*))

(define (all-posts)
  (reverse *posts*))
""" @ lisp

'all-posts'

In [ ]:
"""
(list (post-title (find-post "hello"))
      (map post-title (all-posts)))
""" @ lisp

['Hello, Lisp!', ['Hello, Lisp!']]


.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The data layer is now complete: posts can be created, registered, found by slug, and listed in publication order.

Next, define how one post appears in the index:

```python
"""
(define (render-post-link p)
  (Li
    (A (post-title p)
       (@ (href (string-append "/post/" (post-slug p)))))
    (Span (string-append "  " (post-date p))
          (@ (class "meta")))))
""" @ lisp
```

Test it by rendering every post:

```python
"""
(Ul (map render-post-link (all-posts)))
""" @ lisp
```

This separates **data** from **presentation**: the post remains a tagged list, while `render-post-link` decides how that data becomes HTML.

```json {.usage}
{"model": "gpt-5.6-luna", "prompt_tokens": 43697, "completion_tokens": 316, "total_tokens": 44013, "cached_tokens": 43118, "reasoning_tokens": 144, "cost": 0.0020360400000000002}
```


In [ ]:
"""
(define (render-post-link p)
  (Li
    (A (post-title p)
       (@ (href (string-append "/post/" (post-slug p)))))
    (Span (string-append "  " (post-date p))
          (@ (class "meta")))))
""" @ lisp

'render-post-link'

In [ ]:
"""
(Ul (map render-post-link (all-posts)))
""" @ lisp

KeyError: '@'

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()